In [ ]:
%load_ext autoreload
%autoreload 2
import re
import numpy as np
import read_file, ploting, data_handling, data_mod
import pandas as pd
import os


In [5]:
# import os
# work_path = os.getcwd().split("\\")[:-1]
# work_path = "\\".join(work_path)
# os.chdir(work_path)
# os.getcwd()

'c:\\Users\\elie.tisseur\\Desktop\\testpartage\\Cour Raoult\\Modelisation\\Multi_Hydro\\post_9_mh\\simulation 1m'

## Ajout gullies

In [ ]:
gully_path = r'Inputs/modif/gully_post_9_1m.asc'
luse_path = r'Inputs/modif/luse_post_9_1m.asc'
meta_gully, data_gully = read_file.read_asc_file(gully_path, ignore_first_line=False)
meta_luse, data_luse = read_file.read_asc_file(luse_path, ignore_first_line=True)
# ploting.create_plotly_map(data_gully, meta_gully, "luse_post_9_1m_gully.asc", color="reds")

In [ ]:

data_luse_gully = data_luse.copy()
data_luse_gully[(data_gully==1) & (~np.isnan(data_luse_gully))] = 26
np.nanmax(data_luse_gully)
data_handling.write_asc_file("luse_post_9_1m_gully.asc", meta_gully, data_luse_gully, "luse_post_9_1m_gully.asc")
ploting.create_plotly_map(data_luse_gully, meta_gully, "luse_post_9_1m_gully.asc", color="rainbow")

## Modification drainage input

In [112]:
def get_inflows_nodes(file_path: str, ) -> pd.DataFrame:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist.")
    with open(file_path, 'r',) as file:
        nodes = []
        lire_nodes = False
        for line in file:
            if line.strip() == '[INFLOWS]':
                lire_nodes = True
                continue
            if lire_nodes and line.strip().startswith('['):
                break  # Arrêter à la prochaine section
            if lire_nodes and line.strip() and not line.startswith(';;'):
                nodes.append(line.strip())
    if not nodes:
        raise ValueError("No node coordinates found in the [COORDINATES] section.")
    # Convertion en DataFrame
    # df_inflows = pd.DataFrame([node.split() for node in nodes],)
    # df_inflows = df_inflows.apply(pd.to_numeric)
    # df_inflows["Node"] = df_inflows["Node"].astype(int)
    return [int(node.split()[0]) for node in nodes]

list_gully_nodes = get_inflows_nodes(r"Inputs/modif/reseau_9_post.txt")
print(list_gully_nodes)

17


In [140]:
path_reseau = r"Inputs/modif/reseau_9_post.txt"
cellsize = meta_gully["cellsize"]
nodes_coord = read_file.get_nodes_coord(path_reseau, )
gully_coord = nodes_coord.copy()
gully_coord = gully_coord.loc[gully_coord["Node"].isin(list_gully_nodes), ]
print(gully_coord)
nrows, ncols = data_gully.shape
grid_gully = np.zeros((nrows, ncols))
gully_coord["col"] = ((gully_coord.X - meta_gully["xllcorner"]) / cellsize).astype(int)
gully_coord["row"] = nrows - 1 - np.floor((gully_coord.Y - meta_gully["yllcorner"]) / cellsize).astype(int)

print(gully_coord)
print(meta_gully["xllcorner"], meta_gully["yllcorner"])


# # points qui tombent sur une cellule "gully" (valeur == 1)
# points_dans_gully = gully_coord.loc[gully_coord["valeur_gully"] == 1, "Node"]

# # valeur du raster à la position de chaque point
# gully_coord["valeur_gully"] = data_gully[row.values, col.values]

# # points qui tombent sur une cellule "gully" (valeur == 1)
# points_dans_gully = gully_coord.loc[gully_coord["valeur_gully"] == 1, "Node"]


    Node           X            Y
0    153  620869.103  6876378.456
1    154  620853.270  6876375.871
4    166  620960.263  6876410.743
5    167  620850.564  6876407.595
9    181  620970.150  6876441.473
10   183  620959.935  6876459.854
11   184  620843.445  6876446.568
12   190  620957.936  6876476.535
13   195  620939.794  6876497.077
17   202  620878.370  6876512.370
18   203  620898.511  6876515.009
19   216  620800.720  6876548.926
20   269  620835.845  6876476.291
21   326  620967.293  6876399.594
25   336  620839.732  6876463.059
26   340  620923.710  6876506.117
27   356  620804.032  6876537.508
    Node           X            Y  col  row
0    153  620869.103  6876378.456   79  201
1    154  620853.270  6876375.871   63  203
4    166  620960.263  6876410.743  170  168
5    167  620850.564  6876407.595   61  171
9    181  620970.150  6876441.473  180  138
10   183  620959.935  6876459.854  170  119
11   184  620843.445  6876446.568   53  132
12   190  620957.936  6876476.535  1

In [135]:
np.argwhere(data_gully==1)

array([[ 30,  11],
       [ 41,  14],
       [ 64, 109],
       [ 67,  88],
       [ 73, 134],
       [ 73, 185],
       [ 82, 150],
       [102, 168],
       [103,  46],
       [113, 195],
       [116,  50],
       [119, 170],
       [132,  53],
       [138, 180],
       [168, 170],
       [171,  61],
       [179, 177],
       [201,  79],
       [203,  63]])

In [126]:
print(data_gully)

[[nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]]


## Correction MNT

In [9]:
meta, elev_init = read_file.read_asc_file(file_path="Inputs/modif/elev_9_rgealti_1m.asc",)
ploting.create_plotly_map(elev_init, meta, "Elev correction 0.25m", color="rainbow")

#### Les pixels dont l'écart entre l'élévation la moyenne des évélations des pixels adjacents est supérieur à 0.3m prennent la valeur de la moyenne

In [52]:
pb, elev_cor, nb_iter = data_mod.cor_elev_pb(elev_init, diff_max=0.25, nb_max=0, ignore_center=True)
data_handling.write_asc_file("Inputs/modif/elev_9_rgealti_cor25_1m.asc", metadata=meta, grid=elev_cor, title="Elevation lissee avec moyenne sur (3,3) ecart de 0.25m resolution 1m")

c:\Users\elie.tisseur\PY3.13.2\Lib\site-packages\scipy\ndimage\_filters.py:2420: RuntimeWarning:

Mean of empty slice



Iter  1    11  pixels aberrants
Iter  2    7  pixels aberrants
Iter  3    6  pixels aberrants
Iter  4    3  pixels aberrants
Iter  5    2  pixels aberrants
Iter  6    1  pixels aberrants
Iter  7    0  pixels aberrants
Inputs/modif/elev_9_rgealti_cor25_1m.asc as been created


In [8]:
ploting.create_plotly_map(elev_cor, meta, "Elev correction 0.25m", color="rainbow")

## Modification de la grille d'élévation
#### La grille d'élévation est modifiée en fonction du landuse

In [74]:
dict_luse = read_file.create_dict_luse("Inputs/Surface_input_2018.txt")
print(dict_luse.items())
id = [dict_luse[i]["id"] for i in dict_luse.keys() if dict_luse[i]["name"] == "Forest" ]
import csv
land_config = []
with open("Inputs/land_config.csv", "r") as f:
    data = csv.DictReader(f)
    i=0
    for row in data:
        i += 1
        land_config.append(row)



param_mod_elev = {name: elev for name, elev in zip([land_config[i]["name"] for i in range(len(land_config))], [float(land_config[i]["modif_elev"]) for i in range(len(land_config))])}
param_mod_elev


dict_items([(1, {'id': 1, 'conduction': 1.9e-06, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Forest', 'manning': 0.8, 'intercept_depth': 7.62}), (2, {'id': 2, 'conduction': 1.9e-06, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Grass', 'manning': 0.8, 'intercept_depth': 3.81}), (3, {'id': 3, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Road', 'manning': 0.012, 'intercept_depth': 1.9}), (4, {'id': 4, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Impervious Surface', 'manning': 0.012, 'intercept_depth': 1.9}), (5, {'id': 5, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Toit', 'manning': 0.012, 'intercept_depth': 1.9}), (6, {'id': 6, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Terre', 'manning': 0.8, 'intercept_depth': 1.9}), (7, {'id': 7, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Revetementimper', 'manning': 0.012, 'intercept_depth': 1.9}), 

{'Forest': 0.0,
 'Grass': 0.0,
 'Road': -0.05,
 'Impervious Surface': 0.0,
 'Toit': 5.0,
 'Terre': 0.0,
 'Revetementimper': 0.0,
 'Copeau1': -0.1,
 'Copeau2': -0.1,
 'Copeau3': -0.1,
 'Copeau4': -0.1,
 'Noue1': -0.15,
 'Noue2': -0.15,
 'Noue3': -0.15,
 'Noue4': -0.35,
 'Noue5': -0.15,
 'Noue6': -0.15,
 'Ilotvegetal': -0.05,
 'Toitc1': 6.0,
 'Toitc3': 6.0,
 'Toitc4': 6.0,
 'Toitn1': 6.0,
 'Toitn2': 6.0,
 'Toitn3': 6.0,
 'Toitn6': 6.0,
 'Gully': -0.3}

In [75]:
print(dict_luse)

{1: {'id': 1, 'conduction': 1.9e-06, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Forest', 'manning': 0.8, 'intercept_depth': 7.62}, 2: {'id': 2, 'conduction': 1.9e-06, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Grass', 'manning': 0.8, 'intercept_depth': 3.81}, 3: {'id': 3, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Road', 'manning': 0.012, 'intercept_depth': 1.9}, 4: {'id': 4, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Impervious Surface', 'manning': 0.012, 'intercept_depth': 1.9}, 5: {'id': 5, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Toit', 'manning': 0.012, 'intercept_depth': 1.9}, 6: {'id': 6, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Terre', 'manning': 0.8, 'intercept_depth': 1.9}, 7: {'id': 7, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Revetementimper', 'manning': 0.012, 'intercept_depth': 1.9}, 8: {'id': 8, 'conduction'

In [77]:
elev_grid_mod = elev_cor.copy()
for sol, mod in param_mod_elev.items():
    mask_grid = data_luse_gully == [dict_luse[i]["id"] for i in dict_luse.keys() if dict_luse[i]["name"] == sol][0]
    elev_grid_mod = elev_grid_mod + mod * mask_grid
data_handling.write_asc_file("Inputs/modif/elev_post_9_1m.asc", metadata=meta, grid=elev_grid_mod, title="Elevation modifiée")
ploting.create_plotly_map(elev_grid_mod, meta, "Elevation modifiée", color="rainbow")

Inputs/modif/elev_post_9_1m.asc as been created


In [85]:
ploting.create_plotly_map_soil(data_luse_gully,meta_luse, dict_luse, "Land use post 9 1m",palette=['#3a7535', "#FFB428", "#5A5A5A", "#00B1BE", "#13b604", "#ff1707", '#7a807a',
                                                                                                   '#3a7535', "#FFB428", "#5A5A5A", "#00B1BE", "#13b604", "#ff1707", '#7a807a',
                                                                                                   '#3a7535', "#FFB428", "#5A5A5A", "#00B1BE", "#13b604", "#ff1707", '#7a807a',
                                                                                                   '#3a7535', "#FFB428", "#5A5A5A", "#00B1BE", "#13b604", "#ff1707", '#7a807a'])

In [18]:
from io import StringIO
pd.read_xml("info.xml")

,name,code_sol,modif_elev,sfn,infilt_inf,h_subverse,to_SWMM,to_TREX
0,Forest,1,0.00,False,True,False,False,False
1,Grass,2,0.00,False,True,False,False,False
2,Road,3,-0.05,False,False,False,False,False
3,Impervious Surface,4,0.00,False,False,False,False,False
4,Housefree,5,5.00,False,False,False,False,Noue_70


In [ ]:
ploting.create_plotly_map(elev_init, meta ,"Elev init", grids_hover=[elev_init], info_hover=["elev"], color="blues")

In [ ]:
# for i in range (0,2):
#     elev_pb , elev_cor = cor_elev_pb(elev_cor, 0.4)
# elev_pb = elev_pb.astype(int)
elev_pb , elev_cor = cor_elev_pb(elev_cor, 0.2)
elev_pb = elev_pb.astype(int)
ploting.create_plotly_map(elev_pb, meta ,"Différence >0.4m", grids_hover=[elev_cor], info_hover=["elev"])


In [ ]:

A=[[1.,2.,3.],[4.,5.,6.], [7.,8.,9.]]
np.shape(A)
result = ndimage.generic_filter(A, np.nanmean, size=3, mode='constant', cval=np.nan)
mask = np.ones((3,3))
mask[1,1] = 0
result_mask = ndimage.generic_filter(A, np.nanmean,footprint=mask, mode='constant', cval=np.nan)
result, result_mask

In [ ]:
for i in [[]]:
    print("i")
